## Assignment 3

### scraping news article from article_list.csv

In [2]:


# Import required libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [ ]:
# having a look at the article_list csv
article_list = pd.read_csv('/workspaces/ECBS-5147-Data-Engineering-Cloud-Computing-Managed-AI-Services/serverless/project/articles_raw.csv', 
                 encoding='utf-8')
article_list[10:20]

,Source,Title,URL,Language,Unnamed: 4
10,Prothom Alo,BNP rethinks its approach to allies,https://en.prothomalo.com/bangladesh/politics/...,en,NaN
11,Prothom Alo,BNP’s Masuduzzaman quits race over 'security i...,https://en.prothomalo.com/bangladesh/9pkzu86yuv,en,NaN
12,Prothom Alo,"Meeting of 29 party allies, decision to sort t...",https://en.prothomalo.com/bangladesh/politics/...,en,NaN
13,Prothom Alo,66pc say BNP will win the most seats in the el...,https://en.prothomalo.com/bangladesh/jsto3ql5zk,en,NaN
14,Prothom Alo,BNP wants to push ahead with election plans am...,https://en.prothomalo.com/bangladesh/politics/...,en,NaN
15,Prothom Alo,"Candidates busy in campaign, but Reza Kibria d...",https://en.prothomalo.com/bangladesh/politics/...,en,NaN
16,Prothom Alo,Those who seek to exploit crisis are enemies o...,https://en.prothomalo.com/bangladesh/politics/...,en,NaN
17,Prothom Alo,BNP wants to make Tarique Rahman’s return “mem...,https://en.prothomalo.com/bangladesh/politics/...,en,NaN
18,Prothom Alo,BNP in dilemma over implementation of reform r...,https://en.prothomalo.com/bangladesh/politics/...,en,NaN
19,Prothom Alo,Is BNP stepping into trouble?,https://en.prothomalo.com/opinion/op-ed/pmyhye...,en,NaN


In [29]:
# scrape all daily star

all_daily_star = []
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) "
    + " Chrome/39.0.2171.95 Safari/537.36"
}

for i, url in enumerate(article_list["URL"][:10]):
    
    star_response = requests.get(url, headers=headers)
    daily_star = BeautifulSoup(star_response.content, "html.parser")
    
    title_star = daily_star.find("h1").get_text(strip= True)
    star_body = daily_star.find("article")
    star_paragraphs = star_body.find_all("p")
    star_text = "\n".join([p.get_text(strip=True) for p in star_paragraphs])
    all_daily_star.append(star_text)


In [30]:
# check if it worked
print(all_daily_star[0][:500]) # 0 for the first article
print(all_daily_star[3][:500]) # 3 for the third article

The interim government's revised election timeline with certain conditions has stirred cautious optimism as well as raised questions among  political parties.
While many parties including the BNP have welcomed the proposed  timeline -- mid-February next year --  concerns have emerged regarding how the decision was reached through discussions with only one party.
The BNP had long been demanding that the election be held by the year-end. It had been criticising the interim government for not unvei
Even though a week has passed since the announcement of the election schedule, BNP allies still do not know how many seats they might be allotted.
They said frustration is deepening within their ranks as the BNP has kept the seat-sharing issue hanging for so long.
The parties that were part of the simultaneous movement with the BNP also claimed the BNP did not hold any discussions with them before unveiling its nominees in most constituencies.
In efforts to assuage the allies, the BNP held one-

In [31]:
# scrape all prothomalo

all_prothomalo = []

for i, url in enumerate(article_list["URL"][10:]):
    
    protho_response = requests.get(url, headers=headers)
    prothomalo = BeautifulSoup(protho_response.content, "html.parser")
    
    prothomalo_title = prothomalo.find("h1").get_text(strip= True)
    prothomalo_body = prothomalo.find_all("div", class_="story-element-text")
    prothomalo_text = "\n".join([p.get_text(strip=True) for p in prothomalo_body])

    all_prothomalo.append(prothomalo_text)


In [32]:
# check if it worked
print(all_prothomalo[0][:500])
print(all_prothomalo[5][:500])
print(all_prothomalo[15][:500])

The BNP’s earlier thinking on forming an alliance or contesting elections jointly with its partners in the simultaneous movement is undergoing some change.Relevant responsible sources say the BNP will not form an electoral alliance with its partner parties in the election. There will, however, be seat-sharing arrangements in some constituencies. As part of this, the BNP will not nominate candidates in the constituencies of several top leaders of allied parties. In addition, if the BNP has alread
In the chilly winter weather, electioneering has gained momentum in Habiganj’s four constituencies ahead of the 13th parliamentary election. Nominated candidates from BNP, Jamaat-e-Islami, and other parties are taking part daily in rallies, processions, courtyard meetings, and various campaign programs.Since the election schedule was announced, campaigning has intensified further. Not only town areas, but even remote rural areas of the district are now covered with candidates’ banners, festoons

### Translating from Bengali to English

In [33]:
from pprint import PrettyPrinter
import boto3

print("📚 Setting up the environment...")
pp = PrettyPrinter(indent=2)
translate = boto3.client("translate")
print("✅ Environment setup complete!")
print(f"🌍 Using AWS region: {translate.meta.region_name}")

📚 Setting up the environment...
✅ Environment setup complete!
🌍 Using AWS region: eu-west-1


In [36]:
import sys

print("🔄 Translating from Bengali to English...")
translation = []

for i in all_prothomalo[10:]:
    # Calculate the actual byte size AWS will see
    byte_size = len(i.encode('utf-8'))

    if byte_size > 10000:
        # We split the string into chunks of 3000 characters. 
        # Since Bengali is ~3 bytes/char, 3000 chars is ~9000 bytes (safe)
        chunk_size = 3000
        
        # Split into 3 parts (adjust range if articles are even longer)
        p1 = i[:chunk_size]
        p2 = i[chunk_size:chunk_size*2]
        p3 = i[chunk_size*2:chunk_size*3]
        p4 = i[chunk_size*3:]
        
        # Get translations and extract ['TranslatedText']
        resp1 = translate.translate_text(Text=p1, SourceLanguageCode="bn", TargetLanguageCode="en")
        resp2 = translate.translate_text(Text=p2, SourceLanguageCode="bn", TargetLanguageCode="en")
        
        texts = [resp1['TranslatedText'], resp2['TranslatedText']]
        
        # Only call if p3 actually has text
        if p3.strip():
            resp3 = translate.translate_text(Text=p3, SourceLanguageCode="bn", TargetLanguageCode="en")
            texts.append(resp3['TranslatedText'])
        
        if p4.strip():
            resp4 = translate.translate_text(Text=p4, SourceLanguageCode="bn", TargetLanguageCode="en")
            texts.append(resp4['TranslatedText'])
            
        response = "\n".join(texts)

    else:
        resp = translate.translate_text(Text=i, SourceLanguageCode="bn", TargetLanguageCode="en")
        response = resp['TranslatedText']

    translation.append(response)

# Note: Using pp.pprint might still show some encoding characters in the terminal
#import pprint
# pp = pprint.PrettyPrinter(indent=4)
pp.pprint(translation)

🔄 Translating from Bengali to English...
[ 'With the 13th National Assembly elections ahead, the electoral momentum of '
  'candidates from different parties in the six parliamentary seats in '
  'Sirajganj is steadily gaining momentum. After the announcement of the '
  'schedule, election-centered discussions and debates are also taking place '
  'among the general public. In some seats, the campaign of the rival '
  'candidates prevails, in some places the party clashes. Somewhere there are '
  'discussions about the change of leaders. The electoral heat is increasing '
  'day by day as candidates nominated by different parties take to the field '
  'in the six seats in the district. There has been some party discontent in '
  "the two seats over the BNP's nomination. Former MP Manzoor Kader is in "
  'talks to become the NCP candidate after not receiving the BNP nomination '
  'for the Sirajganj-5 seat. However, the candidates nominated by the BNP have '
  'taken to the field to cam

## translations to dataframe

In [ ]:
# 1. Align the original text with your DataFrame
# This assumes articles_list matches the row order of your CSV (0-9 Daily Star, 10-29 Prothom Alo)
article_list['og_text'] = all_daily_star + all_prothomalo
article_list[:5]

,Source,Title,URL,Language,Unnamed: 4,og_text
0,The Daily Star,New Polls Timing BNP upbeat process irks Jamaa...,https://www.thedailystar.net/news/bangladesh/p...,en,NaN,The interim government's revised election time...
1,The Daily Star,BNP announces candidates for 36 more constitue...,https://www.thedailystar.net/news/bangladesh/p...,en,NaN,BNP has announced its candidates for 36 more c...
2,The Daily Star,Seat-sharing snub riles BNP allies,https://www.thedailystar.net/news/bangladesh/p...,en,NaN,Discontent runs deep among BNP's partners as t...
3,The Daily Star,Seat sharing BNP still keeps allies hanging,https://www.thedailystar.net/news/bangladesh/p...,en,NaN,Even though a week has passed since the announ...
4,The Daily Star,"Election to be tough for BNP, bigger test if v...",https://www.thedailystar.net/news/bangladesh/p...,en,NaN,The next national election will be tough for t...
5,The Daily Star,Old habits die hard? BNP’s response,https://www.thedailystar.net/opinion/views/new...,en,NaN,"An opinion piece titled ""BNP's notes of dissen..."
6,The Daily Star,BNP at 47: Caught between prospects and perils,https://www.thedailystar.net/news/bangladesh/p...,en,NaN,The BNP has survived Sheikh Hasina's 15-year r...
7,The Daily Star,BNP’s show of force on Airport Road and a poli...,https://www.thedailystar.net/opinion/views/new...,en,NaN,The former Prime Minister Khaleda Zia on Tuesd...
8,The Daily Star,Fulfilling Uprising's Aspirations: Correction ...,https://www.thedailystar.net/opinion/editorial...,en,NaN,After having weathered a difficult 15 years in...
9,The Daily Star,Cases against its candidates a concern for BNP,https://www.thedailystar.net/news/bangladesh/p...,en,NaN,The BNP has alerted its election candidates th...


In [38]:

# 2. Define the translation function with your chunking logic
def translate_logic(row):
    text = row['og_text']
    # Use 'Language' (case-sensitive) as per your CSV
    lang = row['Language']
    
    # Logic 1: If English, copy-paste original
    if lang == "en":
        return text
    
    # Logic 2: If Bengali, translate (with chunking if needed)
    if lang == "bn":
        byte_size = len(text.encode('utf-8'))
        
        # Safe size for AWS Translate (10,000 bytes)
        if byte_size <= 10000:
            resp = translate.translate_text(Text=text, SourceLanguageCode="bn", TargetLanguageCode="en")
            return resp['TranslatedText']
        
        else:
            # Chunking logic for longer Bengali articles
            chunk_size = 3000
            # Split text into chunks of 3000 chars
            chunks = [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]
            
            translated_parts = []
            for chunk in chunks:
                if chunk.strip(): # Only translate if there is actual text
                    resp = translate.translate_text(
                        Text=chunk, 
                        SourceLanguageCode="bn", 
                        TargetLanguageCode="en"
                    )
                    translated_parts.append(resp['TranslatedText'])
            
            return "\n".join(translated_parts)
            
    return text  # Fallback for any other cases

# 3. Create the new column by applying the function
print("🔄 Processing and translating articles...")
article_list['translated_text'] = article_list.apply(translate_logic, axis=1)

# 4. Save the results
article_list.to_csv("articles_final.csv", index=False)
print("✅ Done! Saved to articles_final.csv")

🔄 Processing and translating articles...
✅ Done! Saved to articles_final.csv


In [ ]:
# 4. Save the results
#del article_list['Unnamed: 4']
article_list


,Source,Title,URL,Language,og_text,translated_text
0,The Daily Star,New Polls Timing BNP upbeat process irks Jamaa...,https://www.thedailystar.net/news/bangladesh/p...,en,The interim government's revised election time...,The interim government's revised election time...
1,The Daily Star,BNP announces candidates for 36 more constitue...,https://www.thedailystar.net/news/bangladesh/p...,en,BNP has announced its candidates for 36 more c...,BNP has announced its candidates for 36 more c...
2,The Daily Star,Seat-sharing snub riles BNP allies,https://www.thedailystar.net/news/bangladesh/p...,en,Discontent runs deep among BNP's partners as t...,Discontent runs deep among BNP's partners as t...
3,The Daily Star,Seat sharing BNP still keeps allies hanging,https://www.thedailystar.net/news/bangladesh/p...,en,Even though a week has passed since the announ...,Even though a week has passed since the announ...
4,The Daily Star,"Election to be tough for BNP, bigger test if v...",https://www.thedailystar.net/news/bangladesh/p...,en,The next national election will be tough for t...,The next national election will be tough for t...
5,The Daily Star,Old habits die hard? BNP’s response,https://www.thedailystar.net/opinion/views/new...,en,"An opinion piece titled ""BNP's notes of dissen...","An opinion piece titled ""BNP's notes of dissen..."
6,The Daily Star,BNP at 47: Caught between prospects and perils,https://www.thedailystar.net/news/bangladesh/p...,en,The BNP has survived Sheikh Hasina's 15-year r...,The BNP has survived Sheikh Hasina's 15-year r...
7,The Daily Star,BNP’s show of force on Airport Road and a poli...,https://www.thedailystar.net/opinion/views/new...,en,The former Prime Minister Khaleda Zia on Tuesd...,The former Prime Minister Khaleda Zia on Tuesd...
8,The Daily Star,Fulfilling Uprising's Aspirations: Correction ...,https://www.thedailystar.net/opinion/editorial...,en,After having weathered a difficult 15 years in...,After having weathered a difficult 15 years in...
9,The Daily Star,Cases against its candidates a concern for BNP,https://www.thedailystar.net/news/bangladesh/p...,en,The BNP has alerted its election candidates th...,The BNP has alerted its election candidates th...


In [ ]:
# save the results
article_list.to_csv("articles_final.csv", index=False)
print("✅ Done! Saved to articles_final.csv")

